In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from transformers import ViTModel, ViTConfig

class HybridViTEncoder(nn.Module):
    def __init__(self, vit_name='google/vit-base-patch16-224'):
        super().__init__()
        # 1. ResNet Backbone (Extracts local features/skip connections)
        resnet = models.resnet50(pretrained=True)
        self.conv1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.layer1 = resnet.layer1  # Output for Skip Connection 1
        self.layer2 = resnet.layer2  # Output for Skip Connection 2
        self.layer3 = resnet.layer3  # Input to Transformer

        # 2. Vision Transformer (Captures global context)
        self.vit = ViTModel.from_pretrained(vit_name)
        self.embeddings = self.vit.embeddings
        self.encoder = self.vit.encoder

        # 3. Feature Bridge (Aligns ResNet channels to ViT embedding size)
        self.bridge = nn.Conv2d(1024, 768, kernel_size=1)

    def forward(self, x):
        # Local Path
        x1 = self.conv1(x)
        x2 = self.layer1(self.pool(x1)) # Skip 1
        x3 = self.layer2(x2)            # Skip 2
        x4 = self.layer3(x3)            # Features for ViT

        # Global Path (Transforming spatial features to tokens)
        # Reshape for Transformer: (B, 1024, 14, 14) -> (B, 768, 14, 14)
        y = self.bridge(x4)
        B, C, H, W = y.shape
        y = y.flatten(2).transpose(1, 2) # (B, 196, 768)

        # Apply Transformer Attention
        y = self.encoder(y).last_hidden_state

        # Reshape back to spatial dimensions for Decoder
        y = y.transpose(1, 2).reshape(B, C, H, W)

        return y, [x1, x2, x3] # Return hybrid features and skip connections

In [ ]:
class HybridSegmentationModel(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.encoder = HybridViTEncoder()

        # Standard U-Net Decoder blocks (Upsampling + Concat)
        self.up1 = nn.ConvTranspose2d(768, 512, kernel_size=2, stride=2)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.final_conv = nn.Conv2d(256, num_classes, kernel_size=1)

    def forward(self, x):
        # 1. Encode with ResNet-ViT Hybrid
        global_features, skips = self.encoder(x)

        # 2. Decode and Fuse (Example of one upsampling step)
        # Skip[2] is the high-res feature from ResNet Layer 2
        x = self.up1(global_features)
        # Add skip connection here if dimensions match

        x = self.up2(x)
        return self.final_conv(x) # Output pixel-wise mask